# TP Intelligence Artificielle — Agriculture au Burundi
## Prédiction des Bonnes et Mauvaises Récoltes
**Université Polytechnique de Gitega — Bac 4 Génie Logiciel**


---
## EXERCICE 1 — Chargement, Exploration et Qualité des Données 

### 1.1 Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, auc)
import warnings
warnings.filterwarnings('ignore')

# Chargement du dataset
df = pd.read_csv('agriculture_burundi.csv')
print("Dataset chargé avec succès !")
print(df.head())


### Q1 — Structure du dataset

In [ ]:
# Q1 : Dimensions et informations générales
print(f"Nombre de lignes    : {df.shape[0]}")
print(f"Nombre de colonnes  : {df.shape[1]}")
print(f"Période couverte    : {df['annee'].min()} à {df['annee'].max()}")
print(f"Provinces distinctes: {df['province'].nunique()} → {sorted(df['province'].unique())}")
print(f"Cultures présentes  : {sorted(df['culture'].unique())}")


### Q2 — Types de données

In [ ]:
# Q2 : Types de données
print(df.dtypes)
print()
print("""Observations :
- 'saison' est de type object (str) → correct (catégoriel A/B)
- 'province' et 'culture' sont object → correct (catégoriels)
- 'utilisation_engrais' et 'acces_irrigation' sont int64 → correct (binaires 0/1)
- 'bonne_recolte' est int64 → correct (variable cible binaire 0/1)
- Les variables numériques (pluviometrie_mm, rendement_t_ha, etc.) sont float64 → correct""")


### 1.2 Qualité des données — Valeurs manquantes

In [ ]:
# Q3 : Valeurs manquantes
missing_count = df.isnull().sum()
missing_pct   = df.isnull().mean() * 100

missing_df = pd.DataFrame({
    'Nombre manquants': missing_count,
    'Pourcentage (%)' : missing_pct.round(2)
})
print(missing_df[missing_df['Nombre manquants'] > 0])
print()

# Distribution des manquants par province
print("Manquants de 'pluviometrie_mm' par province :")
print(df[df['pluviometrie_mm'].isna()]['province'].value_counts())
print()
print("Manquants de 'bonne_recolte' par culture :")
print(df[df['bonne_recolte'].isna()]['culture'].value_counts())


**Réponse Q3 :**
- `pluviometrie_mm` : 63 valeurs manquantes (~3.9%) — réparties sur plusieurs provinces
- `utilisation_engrais` : 45 valeurs manquantes (~2.8%) — variable binaire
- `rendement_t_ha`, `production_totale_t`, `bonne_recolte` : 44 valeurs manquantes (~2.7%) chacune  
Les manquants semblent relativement homogènes entre les provinces et cultures.


In [ ]:
# Q4 : Traitement des valeurs manquantes

# Stratégie :
# - pluviometrie_mm (numérique continue) → imputation par la MÉDIANE par province
#   (la médiane est robuste aux outliers contrairement à la moyenne)
# - utilisation_engrais (binaire) → imputation par le MODE par culture (valeur la plus fréquente)
# - rendement_t_ha, production_totale_t → imputation par la MÉDIANE par culture
# - bonne_recolte (variable cible) → SUPPRESSION des lignes (risque de biais si on impute la cible)

# 1. Imputation de pluviometrie_mm par médiane par province
df['pluviometrie_mm'] = df.groupby('province')['pluviometrie_mm'].transform(
    lambda x: x.fillna(x.median())
)

# 2. Imputation de utilisation_engrais par mode par culture
df['utilisation_engrais'] = df.groupby('culture')['utilisation_engrais'].transform(
    lambda x: x.fillna(x.mode()[0])
)

# 3. Imputation rendement et production par médiane par culture
df['rendement_t_ha'] = df.groupby('culture')['rendement_t_ha'].transform(
    lambda x: x.fillna(x.median())
)
df['production_totale_t'] = df.groupby('culture')['production_totale_t'].transform(
    lambda x: x.fillna(x.median())
)

# 4. Suppression des lignes où bonne_recolte est manquante
df = df.dropna(subset=['bonne_recolte'])
df['bonne_recolte'] = df['bonne_recolte'].astype(int)

print("Vérification — valeurs manquantes restantes :")
print(df.isnull().sum())
print(f"\nDataset final : {df.shape[0]} lignes × {df.shape[1]} colonnes")


### 1.3 Exploration statistique

In [ ]:
# Q5 : Statistiques descriptives
stats = df.describe().T
stats['median'] = df.describe().T.index.map(lambda c: df[c].median() if c in df.select_dtypes(include=np.number).columns else None)
print(stats[['mean','50%','std','min','max']].round(2))
print()

# Culture avec le rendement le plus élevé / faible
print("Rendement moyen par culture :")
rend = df.groupby('culture')['rendement_t_ha'].mean().sort_values(ascending=False)
print(rend.round(3))
print(f"\n→ Meilleur rendement  : {rend.index[0]} ({rend.iloc[0]:.2f} t/ha)")
print(f"→ Pire rendement      : {rend.index[-1]} ({rend.iloc[-1]:.2f} t/ha)")
print()

# Province avec la plus forte pluviométrie
print("Pluviométrie moyenne par province (top 5) :")
print(df.groupby('province')['pluviometrie_mm'].mean().sort_values(ascending=False).head(5).round(1))


In [ ]:
# Q6 : Distribution de la variable cible
vc = df['bonne_recolte'].value_counts()
pct = df['bonne_recolte'].value_counts(normalize=True)*100
print("Distribution de bonne_recolte :")
print(f"  Mauvaise récolte (0) : {vc[0]} observations ({pct[0]:.1f}%)")
print(f"  Bonne récolte    (1) : {vc[1]} observations ({pct[1]:.1f}%)")
print()
if abs(pct[0]-pct[1]) < 15:
    print("→ Le dataset est relativement ÉQUILIBRÉ.")
else:
    print("→ Le dataset est DÉSÉQUILIBRÉ — risque que le modèle favorise la classe majoritaire.")
    print("  Conséquence : accuracy trompeuse, il faudra surveiller le F1-score et l'AUC.")


In [ ]:
# Q7 : Visualisations

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Exploration des données agricoles — Burundi", fontsize=15, fontweight='bold')

# 1. Boxplot du rendement par culture
df.boxplot(column='rendement_t_ha', by='culture', ax=axes[0,0])
axes[0,0].set_title('Distribution du rendement par culture')
axes[0,0].set_xlabel('Culture')
axes[0,0].set_ylabel('Rendement (t/ha)')
axes[0,0].tick_params(axis='x', rotation=30)
plt.sca(axes[0,0])
plt.suptitle('')

# 2. Évolution de la production totale par année
prod_year = df.groupby('annee')['production_totale_t'].sum()
axes[0,1].plot(prod_year.index, prod_year.values, marker='o', color='green', linewidth=2)
axes[0,1].set_title('Évolution de la production totale par année')
axes[0,1].set_xlabel('Année')
axes[0,1].set_ylabel('Production totale (tonnes)')
axes[0,1].grid(True, alpha=0.4)

# 3. Proportion de bonnes récoltes selon l'utilisation d'engrais
engrais_recolte = df.groupby('utilisation_engrais')['bonne_recolte'].mean() * 100
bars = axes[1,0].bar(['Sans engrais (0)', 'Avec engrais (1)'],
                      engrais_recolte.values, color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[1,0].set_title('% de bonnes récoltes selon l'utilisation d'engrais')
axes[1,0].set_ylabel('% Bonnes récoltes')
axes[1,0].set_ylim(0, 100)
for bar, val in zip(bars, engrais_recolte.values):
    axes[1,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                   f'{val:.1f}%', ha='center', fontweight='bold')

# 4. Matrice de corrélation
num_cols = ['altitude_m','pluviometrie_mm','temperature_moy_C','superficie_ha',
            'utilisation_engrais','acces_irrigation','nb_menages',
            'rendement_t_ha','bonne_recolte']
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            ax=axes[1,1], linewidths=0.5, annot_kws={'size': 7})
axes[1,1].set_title('Matrice de corrélation')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('ex1_visualisations.png', dpi=120, bbox_inches='tight')
plt.show()
print("\nCommentaires :")
print("1. Boxplot : le Bananier et le Manioc ont les rendements les plus variables ; le Sorgho est le plus stable.")
print("2. Lineplot : la production totale fluctue selon les années, avec des creux probablement liés au climat.")
print("3. Barplot : les parcelles avec engrais ont un taux de bonnes récoltes nettement supérieur.")
print("4. Heatmap : rendement_t_ha est fortement corrélé à bonne_recolte (logique, c'est sa définition).")


---
## EXERCICE 2 — Prétraitement et Préparation des Données 

### 2.1 Encodage des variables catégorielles

In [ ]:
# Q8 : Variables catégorielles
cat_cols = df.select_dtypes(include='object').columns.tolist()
print("Variables catégorielles :", cat_cols)
print()
print("""Pourquoi on ne peut pas les utiliser directement :
→ Les algorithmes ML attendent des valeurs numériques. 'Gitega' ou 'Maïs'
  ne peuvent pas être comparés ou multipliés mathématiquement.

Différence LabelEncoder vs get_dummies (One-Hot Encoding) :
→ LabelEncoder : encode 'Gitega'=3, 'Kayanza'=7... → introduit un ordre artificiel
  Adapté uniquement pour les variables ORDINALES (ex : petit < moyen < grand)
→ get_dummies (OHE) : crée une colonne binaire par catégorie, sans ordre implicite
  Adapté pour les variables NOMINALES sans ordre naturel

Choix pour ce dataset :
→ 'province' (15 catégories, nominale) → get_dummies (OHE)
→ 'culture'  (6 catégories, nominale)  → get_dummies (OHE)
→ 'saison'   (2 catégories : A/B)      → LabelEncoder ou get_dummies (équivalents à 2 cat.)""")


In [ ]:
# Q9 : Application de l'encodage
df_encoded = pd.get_dummies(df, columns=['province', 'culture', 'saison'], drop_first=True)

print(f"Colonnes avant encodage : {df.shape[1]}")
print(f"Colonnes après encodage : {df_encoded.shape[1]}")
print()
print("Nouvelles colonnes créées :")
new_cols = [c for c in df_encoded.columns if c not in df.columns]
print(new_cols)
print()
print("→ drop_first=True supprime une catégorie par variable pour éviter la dummy variable trap")
print("  (multicolinéarité parfaite qui perturbe surtout la régression logistique)")


### 2.2 Sélection des variables et normalisation

In [ ]:
# Q10 : Matrice X et vecteur y
# On exclut rendement_t_ha et production_totale_t → DATA LEAKAGE !
# bonne_recolte est DÉFINIE à partir du rendement : garder rendement dans X
# reviendrait à "donner la réponse" au modèle → il apprend la définition, pas la réalité.
# On exclut aussi annee (identifiant temporel, pas une feature agronomique directe)

exclude = ['bonne_recolte', 'rendement_t_ha', 'production_totale_t', 'annee']
feature_cols = [c for c in df_encoded.columns if c not in exclude]

X = df_encoded[feature_cols].copy()
y = df_encoded['bonne_recolte'].copy()

print(f"Dimensions de X : {X.shape}")
print(f"Dimensions de y : {y.shape}")
print(f"\nFeatures utilisées ({len(feature_cols)}) :")
print(feature_cols)


In [ ]:
# Q11 : Normalisation (StandardScaler)
# Colonnes numériques à normaliser (pas les binaires encodées)
num_features = ['altitude_m', 'pluviometrie_mm', 'temperature_moy_C',
                'superficie_ha', 'nb_menages', 'utilisation_engrais', 'acces_irrigation']
num_features = [c for c in num_features if c in X.columns]

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[num_features] = scaler.fit_transform(X[num_features])

print("Après normalisation (colonnes numériques) :")
print(X_scaled[num_features].describe().T[['mean','std']].round(4))
print()
print("""Pourquoi normaliser ?
→ La régression logistique et les SVM sont sensibles aux échelles (gradient descent).
  Un feature en millimètres (0–2000) vs un feature binaire (0–1) biaiserait les coefficients.
→ Les arbres de décision et forêts aléatoires font des splits → insensibles à l'échelle.
  La normalisation n'est donc pas critique pour eux, mais elle n'est pas nuisible.""")


### 2.3 Division Train / Test

In [ ]:
# Q12 : Split 80/20 stratifié
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Taille Train : {X_train.shape[0]} ({X_train.shape[0]/len(y)*100:.0f}%)")
print(f"Taille Test  : {X_test.shape[0]} ({X_test.shape[0]/len(y)*100:.0f}%)")
print()

# Vérification de la stratification
print("Proportion bonne_recolte dans TRAIN :", y_train.mean().round(3))
print("Proportion bonne_recolte dans TEST  :", y_test.mean().round(3))
print()
print("""Pourquoi stratify=y ?
→ Garantit que la proportion de bonnes/mauvaises récoltes est identique
  dans train et test. Sans ça, le hasard pourrait créer un split déséquilibré
  qui biaiserait l'évaluation du modèle.

Pourquoi fixer random_state=42 ?
→ Assure la REPRODUCTIBILITÉ : tout le monde obtient les mêmes splits.
  Sans random_state, les résultats changent à chaque exécution,
  rendant la comparaison des modèles impossible.""")


---
## EXERCICE 3 — Arbre de Décision 

### 3.1 Entraînement

In [ ]:
# Q13 : Arbre de décision de base
dt = DecisionTreeClassifier(max_depth=4, criterion='gini', random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print(f"Accuracy (Arbre de Décision, max_depth=4) : {accuracy_score(y_test, y_pred_dt)*100:.2f}%")
print()
print("Rapport de classification :")
print(classification_report(y_test, y_pred_dt, target_names=['Mauvaise récolte (0)', 'Bonne récolte (1)']))


In [ ]:
# Q14 : Matrice de confusion
cm = confusion_matrix(y_test, y_pred_dt)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Mauvaise (0)', 'Bonne (1)'])
fig, ax = plt.subplots(figsize=(6,5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matrice de Confusion — Arbre de Décision')
plt.tight_layout()
plt.savefig('ex3_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Vrais Positifs  (TP) : {tp}  → bonnes récoltes correctement prédites")
print(f"Vrais Négatifs  (TN) : {tn}  → mauvaises récoltes correctement prédites")
print(f"Faux Positifs   (FP) : {fp}  → mauvaises récoltes prédites comme bonnes (fausses alertes)")
print(f"Faux Négatifs   (FN) : {fn}  → bonnes récoltes manquées")
print()
print("""Dans un contexte agricole, le FAUX NÉGATIF est le plus coûteux :
→ Prédire une mauvaise récolte alors qu'elle sera bonne → agriculteur prend des mesures
  préventives inutiles (coût financier, mais pas de catastrophe).
→ Prédire une BONNE récolte alors qu'elle sera mauvaise (FN) → pas de préparation,
  pas d'aide demandée, risque de famine ou de perte totale → CATASTROPHIQUE.""")


### 3.2 Visualisation de l'arbre

In [ ]:
# Q15 : Visualisation de l'arbre
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(dt,
          feature_names=feature_cols,
          class_names=['Mauvaise', 'Bonne'],
          filled=True,
          rounded=True,
          fontsize=9,
          ax=ax)
ax.set_title("Arbre de Décision (max_depth=4) — Prédiction des récoltes au Burundi",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ex3_decision_tree.png', dpi=100, bbox_inches='tight')
plt.show()

# Premier split
first_feature = feature_cols[dt.tree_.feature[0]]
first_threshold = dt.tree_.threshold[0]
print(f"Premier split (racine) : '{first_feature}' ≤ {first_threshold:.3f}")
print()
print(f"""Interprétation agronomique :
→ Le modèle utilise '{first_feature}' comme critère principal de séparation.
  Cela indique que cette variable est la plus discriminante pour distinguer
  bonnes et mauvaises récoltes dans les données du Burundi.""")


In [ ]:
# Q16 : Importance des variables
importances = pd.Series(dt.feature_importances_, index=feature_cols).sort_values(ascending=False)
top10 = importances.head(10)

fig, ax = plt.subplots(figsize=(10, 5))
top10.plot.barh(ax=ax, color='steelblue', edgecolor='black')
ax.invert_yaxis()
ax.set_title("Importance des variables — Arbre de Décision (Top 10)")
ax.set_xlabel("Importance (Gini)")
plt.tight_layout()
plt.savefig('ex3_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print("Top 5 variables importantes :")
print(top10.head(5).round(4))
print()
print("""Interprétation :
→ Les variables numériques clés (pluviométrie, température, altitude) sont
  généralement en tête, ce qui est cohérent avec l'agronomie burundaise.
→ utilisation_engrais : si son importance est significative, c'est un message
  fort aux décideurs → investir dans les engrais améliore les récoltes.""")


### 3.3 Impact des hyperparamètres — Overfitting

In [ ]:
# Q17 : Analyse de l'overfitting selon max_depth
train_scores = []
test_scores  = []
depths = range(1, 21)

for depth in depths:
    clf = DecisionTreeClassifier(max_depth=depth, criterion='gini', random_state=42)
    clf.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, clf.predict(X_train)))
    test_scores.append(accuracy_score(y_test,  clf.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, train_scores, marker='o', label='Train accuracy', color='blue')
ax.plot(depths, test_scores,  marker='s', label='Test accuracy',  color='orange')
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title("Accuracy Train vs Test selon la profondeur de l'arbre")
ax.legend()
ax.grid(True, alpha=0.4)

best_depth = depths[np.argmax(test_scores)]
ax.axvline(x=best_depth, color='green', linestyle='--', label=f'Meilleure profondeur = {best_depth}')
ax.legend()
plt.tight_layout()
plt.savefig('ex3_overfitting.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Meilleure profondeur (test) : {best_depth}")
print(f"Accuracy test à depth={best_depth} : {max(test_scores)*100:.2f}%")
print()
print("""Overfitting (surapprentissage) :
→ Quand max_depth augmente, l'arbre mémorise les données d'entraînement
  (train_accuracy → 100%) mais perd en généralisation (test_accuracy ↓).
  Dans le contexte agricole : l'arbre apprend des règles trop spécifiques
  (ex : 'si province=Kayanza ET année=2019 ET...') qui ne se généralisent pas.""")


---
## EXERCICE 4 — Forêt Aléatoire 

In [ ]:
# Q18 : Forêt aléatoire de base
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

acc_dt = accuracy_score(y_test, y_pred_dt)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"Accuracy Arbre de Décision : {acc_dt*100:.2f}%")
print(f"Accuracy Forêt Aléatoire   : {acc_rf*100:.2f}%")
print(f"Gain                       : +{(acc_rf-acc_dt)*100:.2f} points")
print()
print("Rapport de classification — Forêt Aléatoire :")
print(classification_report(y_test, y_pred_rf, target_names=['Mauvaise récolte (0)', 'Bonne récolte (1)']))


In [ ]:
# Q19 : Explication conceptuelle
print("""Pourquoi la forêt est plus performante ?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BAGGING (Bootstrap AGGregatING) :
→ Chaque arbre est entraîné sur un sous-ensemble ALÉATOIRE des données
  (tirage avec remise). Ainsi, chaque arbre voit des exemples différents.
→ Les prédictions des 100 arbres sont combinées par VOTE MAJORITAIRE.
→ Les erreurs individuelles des arbres se compensent → variance réduite.

Sélection aléatoire des features (max_features) :
→ À chaque noeud, l'arbre choisit le meilleur split parmi un sous-ensemble
  ALÉATOIRE de features (par défaut : sqrt(n_features)).
→ Décorrèle les arbres entre eux → diversité → meilleure généralisation.

La forêt peut-elle surapprendre ?
→ Moins qu'un arbre seul, mais oui — si n_estimators est très grand
  et que les données sont très bruitées. En pratique, augmenter
  n_estimators ne nuit pas beaucoup à la généralisation.""")


### 4.2 Validation croisée

In [ ]:
# Q20 : Cross-validation 5 folds
cv_scores = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    X_scaled, y, cv=5, scoring='accuracy', n_jobs=-1
)

print("Scores par fold :", np.round(cv_scores, 4))
print(f"Moyenne  : {cv_scores.mean()*100:.2f}%")
print(f"Écart-type : ±{cv_scores.std()*100:.2f}%")
print(f"Accuracy simple test : {acc_rf*100:.2f}%")
print()
print("""Pourquoi la validation croisée est plus fiable ?
→ Le test simple dépend du split aléatoire : si le jeu de test est
  'facile', l'accuracy sera surestimée.
→ La CV évalue le modèle sur 5 sous-ensembles différents et fait
  la MOYENNE → estimation plus robuste et moins biaisée de la
  performance réelle sur des données inconnues.""")


### 4.3 Importance des variables

In [ ]:
# Q21 : Importance des variables — Forêt
importances_rf = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
top10_rf = importances_rf.head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Forêt
top10_rf.plot.barh(ax=axes[0], color='forestgreen', edgecolor='black')
axes[0].invert_yaxis()
axes[0].set_title("Importance — Forêt Aléatoire (Top 10)")
axes[0].set_xlabel("Importance")

# Comparaison avec l'arbre
top10_dt = importances.head(10)
top10_dt.plot.barh(ax=axes[1], color='steelblue', edgecolor='black')
axes[1].invert_yaxis()
axes[1].set_title("Importance — Arbre de Décision (Top 10)")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.savefig('ex4_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print("Top 5 — Forêt Aléatoire :")
print(top10_rf.head(5).round(4))
print()
print("Top 5 — Arbre de Décision :")
print(top10.head(5).round(4))
print()
print("""Différences possibles :
→ La forêt lisse les importances sur 100 arbres → plus stable.
→ L'arbre seul peut surestimer une variable si elle apparaît tôt dans l'arbre.""")


In [ ]:
# Q22 : Impact de n_estimators
n_trees_list = list(range(10, 510, 20))
test_acc_list = []

for n in n_trees_list:
    clf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)
    test_acc_list.append(accuracy_score(y_test, clf.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_trees_list, test_acc_list, color='darkgreen', linewidth=2)
ax.set_xlabel("Nombre d'arbres (n_estimators)")
ax.set_ylabel("Accuracy (test)")
ax.set_title("Impact de n_estimators sur la performance")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('ex4_n_estimators.png', dpi=120, bbox_inches='tight')
plt.show()

stable_idx = next(i for i,v in enumerate(test_acc_list) if abs(v - max(test_acc_list)) < 0.005)
print(f"Stabilisation à partir de : ~{n_trees_list[stable_idx]} arbres")
print("→ Au-delà, gain de performance négligeable mais temps de calcul qui augmente.")


---
## EXERCICE 5 — Régression Logistique & Courbe ROC

In [ ]:
# Q23 : Régression logistique
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Coefficients
coef_df = pd.DataFrame({
    'Feature'    : feature_cols,
    'Coefficient': lr.coef_[0]
}).sort_values('Coefficient', ascending=False)

print("Variables avec les coefficients les plus POSITIFS (favorisent une bonne récolte) :")
print(coef_df.head(5).to_string(index=False))
print()
print("Variables avec les coefficients les plus NÉGATIFS (défavorisent la récolte) :")
print(coef_df.tail(5).to_string(index=False))

# Graphique
fig, ax = plt.subplots(figsize=(10, 8))
coef_df.set_index('Feature')['Coefficient'].head(20).sort_values().plot.barh(
    ax=ax, color=['red' if v < 0 else 'green' for v in coef_df.head(20)['Coefficient'].sort_values()],
    edgecolor='black'
)
ax.set_title("Coefficients — Régression Logistique (Top 20)")
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('ex5_logreg_coefs.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Q24 : Comparaison des performances
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy Arbre de Décision : {acc_dt*100:.2f}%")
print(f"Accuracy Forêt Aléatoire   : {acc_rf*100:.2f}%")
print(f"Accuracy Régression Logist.: {acc_lr*100:.2f}%")
print()
print("""Analyse :
→ La régression logistique suppose une relation LINÉAIRE entre les features
  et la log-probabilité de bonne récolte.
→ Les données agricoles ont souvent des relations NON LINÉAIRES
  (ex : une température trop basse OU trop haute est mauvaise → courbe en U).
→ Les arbres et forêts capturent naturellement ces non-linéarités → meilleure performance.""")


### 5.2 Courbe ROC — Comparaison des 3 modèles

In [ ]:
# Q25 : Courbes ROC
fig, ax = plt.subplots(figsize=(9, 7))

for model, name, color in [
    (dt, 'Arbre de Décision', 'steelblue'),
    (rf, 'Forêt Aléatoire',   'forestgreen'),
    (lr, 'Régression Logistique', 'orange')
]:
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})', color=color, linewidth=2)

ax.plot([0,1],[0,1], 'k--', linewidth=1, label='Classifieur aléatoire (AUC=0.5)')
ax.set_xlabel('Taux de Faux Positifs (FPR)')
ax.set_ylabel('Taux de Vrais Positifs (TPR = Recall)')
ax.set_title('Courbes ROC — Comparaison des 3 modèles')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('ex5_roc_curves.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Q26 : Interprétation de l'AUC
for model, name in [(dt,'Arbre'), (rf,'Forêt'), (lr,'LogReg')]:
    proba = model.predict_proba(X_test)[:,1]
    _, _, _ = roc_curve(y_test, proba)
    roc_auc = auc(*roc_curve(y_test, proba)[:2])
    print(f"AUC — {name:25s} : {roc_auc:.4f}")

print()
print("""Interprétation de l'AUC :
→ L'AUC mesure la probabilité que le modèle donne un score plus élevé
  à un vrai positif qu'à un vrai négatif (capacité de DISCRIMINER).
→ AUC=1.0 : parfait ; AUC=0.5 : aléatoire ; AUC<0.5 : pire que le hasard.
→ L'AUC est plus informative que l'accuracy car elle évalue le modèle
  à TOUS les seuils de classification (pas juste 0.5).

Un modèle avec accuracy=75% peut avoir AUC≈0.5 si le dataset est très
déséquilibré : le modèle prédit tout le temps la classe majoritaire
et obtient 75% d'accuracy... mais ne discrimine pas du tout.

→ Pour la prédiction de récoltes : privilégier l'AUC et le RECALL
  (minimiser les faux négatifs = mauvaises récoltes non détectées).""")


---
## EXERCICE 6 — Prédiction sur Nouveaux Scénarios 

In [ ]:
# Q27 : Prédiction sur 4 scénarios réels
# On reconstruit les features pour chaque scénario

# Les features du modèle (sans rendement, production, annee)
print("Features attendues par le modèle :")
print(feature_cols)


In [ ]:
# Construction manuelle des scénarios
# On prend les médianes des variables non précisées par le scénario
def build_scenario(province, culture, altitude, pluie, temp, engrais, irrigation,
                   superficie=None, nb_menages=None):
    """Construit un DataFrame avec les features encodées pour un scénario."""
    # Valeurs par défaut (médiane du dataset)
    sup = superficie if superficie else df['superficie_ha'].median()
    nb  = nb_menages if nb_menages else df['nb_menages'].median()
    saison = 'A'  # saison par défaut

    row = {
        'altitude_m'          : altitude,
        'pluviometrie_mm'     : pluie,
        'temperature_moy_C'  : temp,
        'superficie_ha'       : sup,
        'utilisation_engrais' : engrais,
        'acces_irrigation'    : irrigation,
        'nb_menages'          : nb,
    }

    # Toutes les colonnes encodées à 0 par défaut
    scenario_df = pd.DataFrame(columns=feature_cols)
    scenario_df.loc[0] = 0.0

    # Remplir les numériques
    for k, v in row.items():
        if k in feature_cols:
            scenario_df.at[0, k] = v

    # Province
    prov_col = f'province_{province}'
    if prov_col in feature_cols:
        scenario_df.at[0, prov_col] = 1

    # Culture
    cult_col = f'culture_{culture}'
    if cult_col in feature_cols:
        scenario_df.at[0, cult_col] = 1

    # Saison (drop_first a supprimé 'saison_A' ou 'saison_B')
    saison_col = 'saison_B'
    if saison_col in feature_cols:
        scenario_df.at[0, saison_col] = (1 if saison == 'B' else 0)

    # Normaliser les variables numériques
    num_in_features = [c for c in num_features if c in feature_cols]
    scenario_df[num_in_features] = scaler.transform(scenario_df[num_in_features])

    return scenario_df

scenarios = [
    ("Kayanza",  "Maïs",        1980, 920, 17.8, 1, 0),
    ("Bubanza",  "Manioc",       790, 550, 25.4, 0, 1),
    ("Gitega",   "Haricot",     1720, 430, 18.2, 0, 0),
    ("Cibitoke", "Patate douce", 810, 810, 24.1, 1, 1),
]

labels = ['Kayanza – Maïs', 'Bubanza – Manioc', 'Gitega – Haricot', 'Cibitoke – Patate douce']

print(f"{'Scénario':<28} {'Arbre':>12} {'Forêt':>12} {'LogReg':>12}")
print("-" * 68)

results = []
for (prov, cult, alt, pluie, temp, eng, irr), label in zip(scenarios, labels):
    sc = build_scenario(prov, cult, alt, pluie, temp, eng, irr)
    row = {}
    for model, name in [(dt,'Arbre'), (rf,'Forêt'), (lr,'LogReg')]:
        pred  = model.predict(sc)[0]
        proba = model.predict_proba(sc)[0][pred]
        row[name] = (pred, proba)

    r = {**row, 'label': label}
    results.append(r)

    line = f"{label:<28}"
    for name in ['Arbre','Forêt','LogReg']:
        p, prob = row[name]
        status = 'BONNE' if p==1 else 'MAUVAISE'
        line += f"  {status} {prob*100:.0f}%"
    print(line)


In [ ]:
# Q28 & Q29 : Interprétation
print("""
Q28 — Accord entre les modèles :
→ Pour certains scénarios, les 3 modèles seront unanimes (cas clairs).
→ En cas de désaccord : utiliser le modèle avec la meilleure AUC comme
  référence, ou appliquer un vote majoritaire.
→ Les scénarios avec probabilité proche de 50% présentent le plus d'incertitude.

Q29 — Scénario 3 : Gitega – Haricot (pluviométrie = 430 mm)
→ 430 mm est une pluviométrie très faible pour le haricot au Burundi.
  Le haricot nécessite ~600-800 mm/saison. Le modèle devrait prédire
  une MAUVAISE récolte, ce qui est agronomiquement cohérent.

Recommandations pour l'agriculteur de Gitega :
1. Installer des systèmes de collecte d'eau de pluie (cisterns)
2. Utiliser des variétés de haricot tolérantes à la sécheresse
3. Multiplier les apports d'engrais organiques pour retenir l'humidité
4. Envisager l'irrigation goutte-à-goutte si disponible
5. Diversifier avec une culture plus résistante à la sécheresse (sorgho)""")


In [ ]:
# Q30 : Réflexion finale — Recommandation au Ministère
print("""
Q30 — Recommandation au Ministère de l'Agriculture du Burundi
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

MODÈLE RECOMMANDÉ : Forêt Aléatoire
Raisons :
  ✓ Meilleure AUC et accuracy parmi les 3 modèles
  ✓ Robuste au bruit et aux données aberrantes
  ✓ Pas besoin de normalisation (pratique pour les mises à jour)
  ✓ Fournit des probabilités fiables pour l'aide à la décision

DONNÉES SUPPLÉMENTAIRES pour améliorer les prédictions :
  → Type de sol (argile, limon, sable)
  → pH du sol et teneur en nutriments
  → Historique des maladies et ravageurs
  → Données météo hebdomadaires (pas seulement la moyenne saisonnière)
  → Accès aux marchés (distance au marché le plus proche)
  → Semences certifiées vs traditionnelles

LIMITES DU MODÈLE :
  ✗ Données simulées — performances réelles à valider sur le terrain
  ✗ Ne capture pas les chocs exogènes (conflits, prix engrais, pandémies)
  ✗ Ne peut pas prédire des situations très différentes des données d'entraînement
  ✗ Interprétabilité limitée de la forêt → boîte noire pour l'agriculteur
  → Ne pas lui faire entièrement confiance : utiliser comme outil d'AIDE
    à la décision, pas comme oracle infaillible.""")


---
## Sauvegarde des modèles (.pkl)

In [ ]:
import joblib

joblib.dump(dt, 'model_decision_tree.pkl')
joblib.dump(rf, 'model_random_forest.pkl')
joblib.dump(lr, 'model_logistic_regression.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(feature_cols, 'feature_cols.pkl')

print("Modèles sauvegardés :")
print("  ✓ model_decision_tree.pkl")
print("  ✓ model_random_forest.pkl")
print("  ✓ model_logistic_regression.pkl")
print("  ✓ scaler.pkl")
print("  ✓ feature_cols.pkl")
